# paintingReorganize — GPU notebook (v2)

Rearranges a painting's pixels into a smooth palette: **same pixels, same
dense rectangle**, reorganised.  Two stages: a sliding-window sort
(seconds, CPU) then annealed exchange dynamics under a composition field
+ a long-range hollow kernel (GPU).

**First: Runtime → Change runtime type → T4 GPU.**

New in v2: the kernel reaches 40% of the image (pyramid-accelerated
octave ladder ~ 1/r², so distant same-colour regions finally attract and
merge instead of surviving as blobs), and the starting temperature is
calibrated from the energy landscape instead of hard-coded.

In [ ]:
!nvidia-smi -L || echo "NO GPU - Runtime > Change runtime type > T4 GPU"
import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

## Setup

In [ ]:
import os
if not os.path.isdir('repo'):
    !git clone -q --branch claude/brave-newton-xsqgpe https://github.com/ardila/paintingReorganize.git repo
%cd repo
!git pull -q
!pip -q install scipy pillow imageio imageio-ffmpeg
import importlib, numpy as np, torch, time
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
import smooth_palette_gpu as G
importlib.reload(G)          # safe after a git pull in a live runtime
import matplotlib.pyplot as plt
print("ready | kernel default top_frac=0.40, t0='auto'")

## Quick run + benchmark

The log line shows the calibrated T0 and the sigma ladder — on a
full-resolution painting the ladder should reach into the hundreds of
pixels. Cost scales linearly with pixels; use the ms/sweep figure to
predict any run.

In [ ]:
rgb = np.asarray(Image.open('demoiselles.jpg').convert('RGB'))
t = time.time(); out = G.run(rgb, sweeps=300, log_every=100); el = time.time()-t
print(f"{rgb.shape[1]}x{rgb.shape[0]} -> {el/300*1000:.0f} ms/sweep; 6000 sweeps = {el/300*6000/60:.1f} min")
plt.figure(figsize=(7,7)); plt.imshow(out); plt.axis('off'); plt.title('300 sweeps (mid-melt - supposed to look bad!)'); plt.show()

## How temperature works, and how to tune it

The anneal starts hot (deliberately melting the stage-1 seed), cools
geometrically, and finishes with greedy settling. **It gets worse before
it gets better** — roughness peaks at ~2.5x the seed's before dropping
below it. Short runs land in the damaged phase.

T0 is calibrated, not guessed: sample swap proposals at the seed, look at
the uphill ones, and choose T0 so the median uphill proposal is accepted
with probability `accept`. This is dimensionless, so it transfers across
paintings and resolutions (a hard-coded T0 does not — gain magnitudes
scale with colour variance and kernel reach).

`accept` is the one temperature knob left, and it has NOT been carefully
tuned — sweep it here:
- low (0.05): gentle melt, stays close to the seed, may not escape its
  grid-aligned boundaries
- high (0.6): deep melt, more reorganisation, needs the full cool to
  recover

In [ ]:
SRC, SCALE, SWEEPS = 'demoiselles.jpg', 0.5, 2500
im = Image.open(SRC).convert('RGB')
im = im.resize((int(im.width*SCALE), int(im.height*SCALE)), Image.LANCZOS)
rgb = np.asarray(im)

results = []
for acc in (0.05, 0.25, 0.6):
    t = time.time()
    out = G.run(rgb, sweeps=SWEEPS, accept=acc, verbose=False)
    results.append((acc, out, time.time()-t)); print(f"accept={acc}: {time.time()-t:.0f}s")

fig, ax = plt.subplots(1, len(results), figsize=(5*len(results), 5))
for i,(acc,out,el) in enumerate(results):
    ax[i].imshow(out); ax[i].axis('off'); ax[i].set_title(f'accept={acc} ({el:.0f}s)')
plt.tight_layout(); plt.show()

## Parameter sweep: kernel reach and field strength

- `top_frac`: how far a large colour region pulls a small one, as a
  fraction of the short side. Too small leaves mid-scale blobs (this was
  the bug that made full-res Starry Night blotchy).
- `lam`: composition-field strength. 0 collapses to a bullseye; too much
  pinches colour regions apart. 12 was good at ~1 MP with the OLD short
  kernel — with the long-range kernel it may want re-tuning, which is
  exactly what this cell is for.

In [ ]:
SRC, SCALE, SWEEPS = 'demoiselles.jpg', 0.5, 2500
im = Image.open(SRC).convert('RGB')
im = im.resize((int(im.width*SCALE), int(im.height*SCALE)), Image.LANCZOS)
rgb = np.asarray(im)

CONFIGS = [dict(top_frac=f, lam=l) for f in (0.10, 0.40) for l in (6.0, 12.0, 25.0)]
results = []
for cfg in CONFIGS:
    t = time.time()
    out = G.run(rgb, sweeps=SWEEPS, verbose=False, **cfg)
    results.append((cfg, out)); print(f"{cfg}: {time.time()-t:.0f}s")

cols = 3; rows = (len(results)+cols-1)//cols
fig, ax = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
for i,(cfg,out) in enumerate(results):
    a = ax.flat[i]; a.imshow(out); a.axis('off')
    a.set_title(f"top_frac={cfg['top_frac']} lam={cfg['lam']}")
for j in range(len(results), rows*cols): ax.flat[j].axis('off')
plt.tight_layout(); plt.show()

## Always check 1:1 before believing a result — previews hide grain

In [ ]:
cfg, out = results[-2]
y, x = out.shape[0]//2, out.shape[1]//2
plt.figure(figsize=(9,9)); plt.imshow(out[y-200:y+200, x-200:x+200])
plt.title(f'1:1 crop {cfg}'); plt.axis('off'); plt.show()

## Video of the evolution

Melt, then condense. The clearest way to understand the schedule.

In [ ]:
import imageio.v2 as imageio
SRC, SCALE, SWEEPS = 'demoiselles.jpg', 0.5, 4000
im = Image.open(SRC).convert('RGB')
im = im.resize((int(im.width*SCALE), int(im.height*SCALE)), Image.LANCZOS)
rgb = np.asarray(im)

writer = imageio.get_writer('evolution.mp4', fps=30, quality=8, macro_block_size=1)
out = G.run(rgb, sweeps=SWEEPS, verbose=False,
            frame_every=max(1, SWEEPS//600), on_frame=lambda i,f: writer.append_data(f))
for _ in range(60): writer.append_data(out)
writer.close()
from IPython.display import Video, display
display(Video('evolution.mp4', embed=True, width=640))
from google.colab import files; files.download('evolution.mp4')

## Full-quality run (any painting in the repo, or upload your own)

In [ ]:
SRC = 'starry_night.png'
rgb = np.asarray(Image.open(SRC).convert('RGB'))
t = time.time()
out = G.run(rgb, sweeps=6000)
print(f"{(time.time()-t)/60:.1f} min")
Image.fromarray(out).save('result.png')
plt.figure(figsize=(16,10)); plt.imshow(out); plt.axis('off'); plt.show()
from google.colab import files; files.download('result.png')

## Garden with Peacocks (10.6 MP — the reason this notebook exists)

In [ ]:
!wget -q -O peacocks.jpg "https://commons.wikimedia.org/wiki/Special:FilePath/Franti%C5%A1ek_Kupka_%E2%80%93_Garden_with_Peacocks.jpg?width=3609"
rgb = np.asarray(Image.open('peacocks.jpg').convert('RGB'))
print(rgb.shape)
t = time.time()
out = G.run(rgb, sweeps=6000)
print(f"{(time.time()-t)/60:.1f} min")
Image.fromarray(out).save('peacocks_smooth.png')
plt.figure(figsize=(16,13)); plt.imshow(out); plt.axis('off'); plt.show()
from google.colab import files; files.download('peacocks_smooth.png')